# Review the full-sequence candidate on CPU

Use a CPU runtime. This notebook reads the saved logs from Drive after the GPU run has finished. It does not load the model or rerun any training. The run saves request-level results and checkpoints, but not the live per-step progress-bar output.


In [ ]:
import json
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
output = Path("/content/drive/MyDrive/uncle/E08_forgetting_trace/full_sequence_27step_candidate")
stem = "seq1_resnet50_seed0"
paths = {name: output / f"{name}_{stem}.json"
         for name in ("origin", "environment", "history", "costs", "summary")}
files = {name: json.loads(path.read_text(encoding="utf-8"))
         for name, path in paths.items()}
origin, environment = files["origin"], files["environment"]
history, costs, numbers = files["history"], files["costs"], files["summary"]
requests = origin["config"]["requests"]
assert len(history) == len(requests) == 30
assert all([row["action"], row["task"]] == request
           for row, request in zip(history, requests))
assert costs["requests"][-1]["index"] == len(requests) - 1
checkpoint = output / f"checkpoint_{stem}.pt"
first_forget_checkpoint = output / f"checkpoint_{stem}_after_request02_forget3.pt"
assert checkpoint.is_file() and checkpoint.stat().st_size > 0
assert first_forget_checkpoint.is_file() and first_forget_checkpoint.stat().st_size > 0
print("Completed requests:", len(history), "of", len(requests))
print("Code commit:", environment["git_commit"])
print("Source checkpoint SHA-256:", origin["starting_checkpoint_sha256"])
print("Run folder:", output)


## Read the saved result

The short screen checks the first `U3` request. The full-sequence metrics include later requests. These checks are exploratory and do not establish successful unlearning.


In [ ]:
first_forget = history[2]
assert (first_forget["action"], first_forget["task"], first_forget["burn_in"]) == ("forget", "3", 27)
drift = abs(first_forget["after"]["0"] - first_forget["before"]["0"])
first_spill = sum(abs(first_forget["after"][task] - value)
                  for task, value in first_forget["before"].items() if task != "3")
short_screen = first_forget["after"]["3"] <= 13.0 and drift < 5.0
recovery_entry = (numbers["retain_accuracy"] is not None
                  and numbers["mean_spill"] is not None
                  and numbers["retain_accuracy"] > 40.0
                  and numbers["mean_spill"] < 3.0)
print("First U3: task 3", first_forget["after"]["3"],
      "task 0", first_forget["after"]["0"],
      "drift", drift, "spill", first_spill)
print("Exploratory <=13% and <5-point short screen:", short_screen)
print("Full sequence: retain", numbers["retain_accuracy"],
      "forget", numbers["forget_accuracy"],
      "mean spill", numbers["mean_spill"],
      "mean relapse", numbers["mean_relapse"])
print("Recovery-entry gate (>40 retain, <3 mean spill):", recovery_entry)
print("Total GPU hours:", costs["totals"].get("gpu_hours"))
print("Post-U3 checkpoint:", first_forget_checkpoint)


## Inspect every request

The saved history has the accuracy of every seen task before and after each request. This table shows the requested task, spill during each forget request, time, and peak GPU memory. The full JSON files remain in the run folder.


In [ ]:
import pandas as pd

cost_by_index = {row["index"]: row for row in costs["requests"]}
rows = []
for record in history:
    task, index = record["task"], record["index"]
    before, after = record["before"], record["after"]
    spill = (sum(abs(after[name] - value) for name, value in before.items()
                 if name != task) if record["action"] == "forget" else None)
    cost = cost_by_index.get(index, {})
    peak = cost.get("peak_memory_bytes")
    rows.append({"index": index, "action": record["action"], "task": task,
                 "target_before": before.get(task), "target_after": after[task],
                 "spill": spill, "seconds": cost.get("seconds"),
                 "peak_GiB": peak / 2**30 if peak is not None else None})
display(pd.DataFrame(rows))
